# Minimax

Using TicTacToe to test out some of the minimax algorithms. Even though one can explore the entire tree of the game, restricting the tree to certain depth artificially.

In [1]:
class TicTacToe:
    winning_positions = [
        {(0, 0), (0, 1), (0, 2)},
        {(1, 0), (1, 1), (1, 2)},
        {(2, 0), (2, 1), (2, 2)},
        {(0, 0), (1, 0), (2, 0)},
        {(0, 1), (1, 1), (2, 1)},
        {(0, 2), (1, 2), (2, 2)},
        {(0, 0), (1, 1), (2, 2)},
        {(0, 2), (1, 1), (2, 0)},
    ]

    def to_string(self):
        result = ""
        for i in range(3):
            for j in range(3):
                chosen = (i, j)
                if chosen in self.x:
                    result += 'X '
                elif chosen in self.o:
                    result += 'O '
                else:
                    result += '. '
            result += '\n'
        return result

    def max_evaluation(self):
        return 10

    def __init__(self, x= set(), o= set(), turn='x'):
        self.x = x
        self.o = o
        self.turn = turn

    def is_max(self):
        return self.turn == 'x'
    
    def depth(self):
        return len(self.x) + len(self.o)

    def children(self):
        for i in range(3):
            for j in range(3):
                chosen = (i, j)
                if chosen not in self.x and chosen not in self.o:
                    if self.turn == 'x':
                        yield TicTacToe(self.x | {chosen}, self.o, 'o')
                    else:
                        yield TicTacToe(self.x, self.o | {chosen}, 'x')

    def is_won(self):
        if self.depth() == 9:
            return '-'
        for positions in TicTacToe.winning_positions:
            if self.x.issuperset(positions):
                return 'x'
            if self.o.issuperset(positions):
                return 'o'
            
        return None
    
    def evaluate(self):
        if self.is_won() == 'x':
            return 10
        elif self.is_won() == 'o':
            return -10
        elif self.is_won() == '-':
            return 0
        else:
            x_count = 0
            o_count = 0

            for positions in TicTacToe.winning_positions:
                if positions.isdisjoint(self.o):
                    x_count += 1
                if positions.isdisjoint(self.x):
                    o_count += 1
            
            return x_count - o_count



## Pure Minimax

Complete tree exploration and evaluate the perfect move.

In [2]:
def minimax(node):
    if node.is_won():
        return node.evaluate()
    if node.is_max():
        return max(minimax(child) for child in node.children())
    else:
        return min(minimax(child) for child in node.children())
    
minimax(TicTacToe())

0

## Alpha Beta Pruning

Alpha - What's the best the max player can do.

Beta - What's the best the min player can do.

If Alpha >= Beta, neither of them are going to take the game, we cut off.

In [3]:
def alpha_beta(node, alpha=-float('inf'), beta=float('inf'), depth=2):
    if node.is_won() or depth <= 0:
        print( node.to_string(), node.evaluate(), sep="\n", end="\n\n")
        return node.evaluate()
    elif node.is_max():
        for child in node.children():
            alpha = max(alpha, alpha_beta(child, alpha, beta, depth - 1))
            if alpha >= beta:
                return beta  # alpha cut-off
    else:
        for child in node.children():
            beta = min(beta, alpha_beta(child, alpha, beta, depth - 1))
            if alpha >= beta:
                return alpha  # beta cut-off
    return alpha if node.is_max() else beta
            
alpha_beta(TicTacToe(), depth=5)

X O X 
O X . 
. . . 

3

X O X 
O . X 
. . . 

2

X O X 
O . . 
X . . 

2

X O X 
O . . 
. X . 

3

X O X 
O . . 
. . X 

2

X O X 
X O . 
. . . 

1

X O X 
. O X 
. . . 

1

X O X 
. O . 
X . . 

1

X O X 
. O . 
. X . 

2

X O X 
. O . 
. . X 

1

X O X 
X . O 
. . . 

2

X O X 
X . . 
O . . 

1

X O X 
. X . 
O . . 

2

X O X 
X . . 
. O . 

3

X O X 
X . . 
. . O 

1

X O X 
. X . 
. . O 

2

X O O 
X X . 
. . . 

2

X O O 
X . X 
. . . 

1

X O O 
X . . 
X . . 

10

X O O 
X . . 
. X . 

2

X O O 
X . . 
. . X 

2

X O X 
X O . 
. . . 

1

X O . 
X O X 
. . . 

0

X O . 
X O . 
X . . 

10

X O X 
X . O 
. . . 

2

X O . 
X X O 
. . . 

2

X O . 
X . O 
X . . 

10

X O X 
X . . 
O . . 

1

X O . 
X X . 
O . . 

1

X O . 
X . X 
O . . 

0

X O . 
X . . 
O X . 

1

X O . 
X . . 
O . X 

1

X O O 
X X . 
. . . 

2

X O O 
. X X 
. . . 

3

X O O 
. X . 
X . . 

3

X O O 
. X . 
. X . 

3

X O O 
. X . 
. . X 

10

X O X 
O X . 
. . . 

3

X O . 
O X X 
. . . 

3

X O . 
O X . 
X . . 


3

## MT SSS*

Variant of SSS* which is bit more easy to understand. Uses a really large bounds first, and if the result is lesser then it's reasonable, if the result is greater then older upper bound is the best one can do.

In [4]:
def mt_sss_star(node, depth=2):
    G = node.max_evaluation() + 1
    
    while True:
        print("Using bounds:", G - 1, G)
        g = alpha_beta_memoized(node, alpha=G - 1, beta=G, depth=depth)
        if g >= G:
            return g # our older upper bound was the best we could do.
        else:
            G = g

def alpha_beta_memoized(node, alpha=-float('inf'), beta=float('inf'), depth=2, cache={}):
    printable = node.to_string()
    if printable in cache:
        return cache[printable]

    if node.is_won() or depth <= 0:
        print(f"Evaluating\n{printable}\n")
        return node.evaluate()
    elif node.is_max():
        for child in node.children():
            alpha = max(alpha, alpha_beta_memoized(child, alpha, beta, depth - 1, cache))
            if alpha >= beta:
                cache[printable] = beta
                return beta  # alpha cut-off
    else:
        for child in node.children():
            beta = min(beta, alpha_beta_memoized(child, alpha, beta, depth - 1, cache))
            if beta <= alpha:
                cache[printable] = beta
                return alpha  # beta cut-off
    result = alpha if node.is_max() else beta
    cache[printable] = result
    return result
            
mt_sss_star(TicTacToe(), depth=10)

Using bounds: 10 11
Evaluating
X O X 
O X O 
X . . 


Evaluating
X O X 
O X O 
O X X 


Evaluating
X O X 
O X O 
. . X 


Evaluating
X O X 
O O X 
X O . 


Evaluating
X O X 
O O X 
O X X 


Evaluating
X O X 
O O X 
. . X 


Evaluating
X O X 
O O O 
X X . 


Evaluating
X O X 
O O O 
X . X 


Evaluating
X O X 
O O O 
. X X 


Evaluating
X O X 
O O X 
. . X 


Evaluating
X O O 
X X O 
X . . 


Evaluating
X O O 
X X O 
O X X 


Evaluating
X O O 
X X O 
. . X 


Evaluating
X O O 
X O X 
X . . 


Evaluating
X O O 
X O X 
O X . 


Evaluating
X O O 
X O X 
O . X 


Evaluating
X O O 
X . . 
X . . 


Evaluating
X O O 
X O . 
X X . 


Evaluating
X O O 
X O O 
X X X 


Evaluating
X O O 
X O . 
X . X 


Evaluating
X O O 
O X X 
X O X 


Evaluating
X O O 
O X X 
O X X 


Evaluating
X O O 
O X X 
. . X 


Evaluating
X O O 
O X O 
X X X 


Evaluating
X O O 
O X . 
X . X 


Evaluating
X O O 
O X . 
. X X 


Evaluating
X O O 
. X . 
. . X 


Evaluating
X O O 
O O X 
X X X 


Evaluating
X O O 
O O X 
X X

10